# Exploratory Data Analysis — Customer Churn

> **Goal**: Understand the dataset, discover patterns driving churn, and extract actionable business insights before building ML models.

**Dataset**: Telco Customer Churn (7,043 customers, 21 features) — real-world data from IBM Sample Datasets.

In [ ]:
# ── Imports & Config ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("Libraries loaded ✓")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────
df = pd.read_csv("../data/raw data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
print("Columns:", list(df.columns))
df.head()

## 1. Dataset Overview

In [ ]:
# Numeric summary
print("── Numeric Features ──")
display(df.describe().round(2))

print("\n── Categorical Features ──")
display(df.describe(include="object"))

In [ ]:
# Missing values check
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing": missing, "Percent": missing_pct})
print("Missing Values:")
display(missing_df[missing_df["Missing"] > 0] if missing.sum() > 0 else "No missing values ✓")

# Duplicate check
print(f"\nDuplicate rows: {df.duplicated().sum()}")

## 2. Target Variable — Churn Distribution

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Countplot
churn_counts = df["Churn"].value_counts()
colors = ["#2ecc71", "#e74c3c"]
ax1 = axes[0]
bars = ax1.bar(churn_counts.index, churn_counts.values, color=colors, edgecolor="black", width=0.5)
for bar, val in zip(bars, churn_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f"{val}\n({val/len(df)*100:.1f}%)", ha="center", fontweight="bold")
ax1.set_title("Churn Distribution", fontsize=14, fontweight="bold")
ax1.set_ylabel("Count")

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, explode=[0, 0.05], shadow=True,
            textprops={"fontsize": 12})
axes[1].set_title("Churn Proportion", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Churn rate: {(df['Churn']=='Yes').mean():.1%}  →  Imbalanced dataset (will need SMOTE)")

## 3. Churn vs Categorical Features

Which customer segments churn the most?

In [ ]:
# Churn by categorical features
cat_features = ["gender", "SeniorCitizen", "Partner", "Dependents",
                "Contract", "InternetService", "PaymentMethod",
                "PaperlessBilling", "TechSupport"]

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    sns.countplot(data=df, x=col, hue="Churn", ax=axes[i],
                  palette={"No": "#2ecc71", "Yes": "#e74c3c"}, edgecolor="black")
    axes[i].set_title(f"Churn by {col}", fontweight="bold")
    axes[i].tick_params(axis="x", rotation=30)
    axes[i].legend(title="Churn", loc="upper right")

plt.suptitle("Churn Distribution Across Categorical Features", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 4. Churn vs Numeric Features

In [ ]:
# KDE plots for numeric features by churn
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numeric_features):
    for label, color in [("No", "#2ecc71"), ("Yes", "#e74c3c")]:
        subset = df[df["Churn"] == label][col].dropna()
        sns.kdeplot(subset, ax=axes[i], label=label, color=color, fill=True, alpha=0.3)
    axes[i].set_title(f"{col} Distribution by Churn", fontweight="bold")
    axes[i].legend(title="Churn")

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Encode for correlation analysis
df_encoded = df.copy()
df_encoded["Churn"] = df_encoded["Churn"].map({"Yes": 1, "No": 0})

# Encode binary columns
binary_map = {"Yes": 1, "No": 0, "Male": 1, "Female": 0}
for col in df_encoded.select_dtypes(include="object").columns:
    if set(df_encoded[col].unique()).issubset({"Yes", "No", "Male", "Female"}):
        df_encoded[col] = df_encoded[col].map(binary_map)

# Keep only numeric columns for correlation
numeric_df = df_encoded.select_dtypes(include=[np.number])

# Correlation with Churn (sorted)
churn_corr = numeric_df.corr()["Churn"].drop("Churn").sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart of correlations with Churn
colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in churn_corr.values]
churn_corr.plot(kind="barh", ax=axes[0], color=colors, edgecolor="black")
axes[0].set_title("Feature Correlation with Churn", fontweight="bold")
axes[0].axvline(x=0, color="black", linewidth=0.8)

# Heatmap of top correlated features
top_features = list(churn_corr.abs().nlargest(10).index) + ["Churn"]
sns.heatmap(numeric_df[top_features].corr(), annot=True, fmt=".2f",
            cmap="RdYlGn_r", center=0, ax=axes[1], square=True)
axes[1].set_title("Correlation Heatmap (Top Features)", fontweight="bold")

plt.tight_layout()
plt.show()

## 6. Key Churn Insights — Drill Down

In [ ]:
# Churn rates by key segments
def churn_rate_by(col):
    return df.groupby(col)["Churn"].apply(lambda x: (x == "Yes").mean()).round(3) * 100

print("── Churn Rate by Contract Type ──")
display(churn_rate_by("Contract").to_frame("Churn Rate (%)").sort_values("Churn Rate (%)", ascending=False))

print("\n── Churn Rate by Internet Service ──")
display(churn_rate_by("InternetService").to_frame("Churn Rate (%)").sort_values("Churn Rate (%)", ascending=False))

print("\n── Churn Rate by Payment Method ──")
display(churn_rate_by("PaymentMethod").to_frame("Churn Rate (%)").sort_values("Churn Rate (%)", ascending=False))

# Tenure buckets
df["TenureBucket"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72],
                             labels=["0-12 mo", "13-24 mo", "25-48 mo", "49-72 mo"])
print("\n── Churn Rate by Tenure Bucket ──")
display(churn_rate_by("TenureBucket").to_frame("Churn Rate (%)").sort_values("Churn Rate (%)", ascending=False))
df.drop(columns=["TenureBucket"], inplace=True)

In [ ]:
# Interactive plot — Churn rate by Contract x Internet Service
churn_pivot = df.groupby(["Contract", "InternetService"])["Churn"].apply(
    lambda x: (x == "Yes").mean() * 100
).reset_index()
churn_pivot.columns = ["Contract", "InternetService", "Churn Rate (%)"]

fig = px.bar(churn_pivot, x="Contract", y="Churn Rate (%)", color="InternetService",
             barmode="group", title="Churn Rate by Contract Type & Internet Service",
             color_discrete_sequence=px.colors.qualitative.Set2,
             text="Churn Rate (%)")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_title="Churn Rate (%)", height=450)
fig.show()

## 7. Summary — Key Business Findings

| # | Insight | Impact |
|---|---------|--------|
| 1 | **Month-to-month contracts** have ~3× higher churn than long-term contracts | Offer discounts for annual plans |
| 2 | **Fiber optic** customers churn more (possibly due to higher cost/competition) | Review fiber pricing strategy |
| 3 | **New customers (tenure < 12 months)** are the highest-risk segment | Focus onboarding & early engagement |
| 4 | **Electronic check** payment users churn significantly more | Incentivize auto-pay enrollment |
| 5 | Customers **without tech support / online security** churn more | Bundle support services |
| 6 | **Senior citizens** have a slightly higher churn rate | Targeted retention programs |
| 7 | The dataset is **imbalanced (~27% churn)** | Use SMOTE during model training |

These insights will guide feature selection and business recommendations in the modeling phase.